In [100]:
!pip install torch transformers peft accelerate psutil llama-cpp-python sentence-transformers pandas -q


In [101]:
import os
os.environ["TOKENIZERS_PARALLELISM"]= "false"

In [102]:
import time
import torch
import psutil
import pandas as pd
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer, util

In [108]:
GGUF_MODEL = "./model-q8_0.gguf"

PROMPTS = [
    """### Instruction:
Answer the HR question accurately.

### Input:
What is HR digitalization?

### Response:
""",

    """### Instruction:
Answer the HR question accurately.

### Input:
Why is succession planning important?

### Response:
""",

    """### Instruction:
Answer the HR question accurately.

### Input:
What is employee engagement?

### Response:
""",

    """### Instruction:
Answer the HR question accurately.

### Input:
What is performance management?

### Response:
""",

    """### Instruction:
Answer the HR question accurately.

### Input:
What is the role of HR in talent acquisition?

### Response:
"""
]
GROUND_TRUTH = [
    "HR digitalization is the use of digital tools and technologies to automate and improve HR processes such as recruitment, payroll, performance management, employee records, and employee engagement.",

    "Succession planning is important because it ensures business continuity by preparing employees to fill key roles, reduces leadership gaps, and supports long-term organizational stability and growth.",

    "Employee engagement refers to the emotional commitment employees have toward their organization, which influences their motivation, productivity, job satisfaction, and willingness to contribute to organizational goals.",

    "Performance management is a continuous process of setting goals, providing feedback, coaching, and evaluating employee performance to improve individual effectiveness and overall organizational outcomes.",

    "The role of HR in talent acquisition is to attract, screen, hire, and onboard suitable candidates while aligning recruitment strategies with business goals and workforce planning needs.",

    "Stay bonuses are financial or non-financial incentives offered to employees to encourage them to remain with an organization for a specified period, often used during mergers, restructuring, or critical project phases.",

    "Retention rates are calculated by dividing the number of employees who remain with the company over a specific period by the total number of employees at the start of that period, then multiplying by 100.",

    "When employees feel valued, it builds trust, increases morale, improves engagement, boosts productivity, and reduces turnover, leading to better performance and a healthier workplace culture.",

    "Onboarding is the structured process of integrating new hires into an organization by providing orientation, training, resources, and support to help them become productive and engaged employees.",

    "Workplace diversity refers to the inclusion of people from different backgrounds, cultures, genders, ages, abilities, and perspectives, which helps foster innovation, fairness, and better decision-making."
]


In [109]:
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")


In [110]:
def get_vram():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0

def accuracy(preds, refs):
    p_emb = embedder.encode(preds, convert_to_tensor=True)
    r_emb = embedder.encode(refs, convert_to_tensor=True)

    sims = util.cos_sim(p_emb, r_emb)

    per_sample_scores = sims.diag().cpu().numpy()

    # for i, score in enumerate(per_sample_scores):
    #     print(f"Sample {i+1} Similarity: {score:.3f}")

    mean_score = per_sample_scores.mean()

    return mean_score

In [118]:
def benchmark_gguf(label):
    llm = Llama(model_path=GGUF_MODEL, n_ctx=2048, n_threads=8, verbose=False)

    outputs = []
    start = time.time()

    for p in PROMPTS:
        response = ""
        stream = llm(p, max_tokens=256, stream=True)

        for output in stream:
            token = output["choices"][0]["text"]
            response += token
            # print(token, end="", flush=True)

        # print("\n \n")
        outputs.append(response)

    end = time.time()

    tokens = sum(len(o.split()) for o in outputs)
    tps = tokens / (end - start)
    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps, 2),
        "Latency(s)": round(end - start, 2),
        "VRAM(MB)": 0,
        "Accuracy": round(acc, 3)
    }

results = []

results.append(benchmark_gguf("GGUF Q8 llama.cpp"))

In [119]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

import threading

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FT_MODEL = "./merged_model"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_PATH = "results.csv"

def benchmark_hf(model_path, label):

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map=DEVICE)

    outputs = []
    start = time.time()

    for prompt in PROMPTS:
        streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

        generation_kwargs = dict(
            **inputs,
            max_new_tokens=256,
            streamer=streamer,
            pad_token_id=tokenizer.eos_token_id
        )

        thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        response = ""
        for token in streamer:
            response += token
        outputs.append(response)
        thread.join()


    end = time.time()

    total_tokens = sum(len(tokenizer.encode(r)) for r in outputs)
    duration = end - start
    tps = total_tokens / duration
    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps, 2),
        "Latency(s)": round(duration, 2),
        "VRAM(MB)": round(get_vram(), 2),
        "Accuracy": round(acc, 3)
    }
results.append(benchmark_hf(BASE_MODEL, "Base Model"))
results.append(benchmark_hf(FT_MODEL, "Fine-tuned"))

df = pd.DataFrame(results)
df.to_csv(RESULTS_PATH, index=False)

print(df)

               Model  Tokens/sec  Latency(s)  VRAM(MB)  Accuracy
0  GGUF Q8 llama.cpp        3.54       99.41      0.00     0.901
1         Base Model       54.39        7.87   5050.44     0.860
2         Fine-tuned       55.54        7.71   5050.44     0.860
